# **G2P Evaluation of a ByT5 Model on Wiktionary Phonemic Data**

This notebook tests a neural G2P model,
[`lowestofthelow/filipino-byt5-g2p`](https://huggingface.co/lowestofthelow/filipino-byt5-g2p), on
the same held-out test split as `Wik_eval.ipynb`. That makes its scores directly comparable with
the WFST model behind the `tagalog_g2p.py` command-line tool.

The model is a ByT5 transformer fine-tuned from CharsiuG2P
(`charsiu/g2p_multilingual_byT5_small_100`). It reads the word one byte at a time and writes the
IPA transcription the same way. Its model card is empty, so the input format below was worked out
by testing, and it's unknown what data it was fine-tuned on. If its training data overlaps with
the Wiktionary test words, the scores here are optimistic.

Run `Wik_eval.ipynb` first: it writes the test split (`dicts/cwik_test.tsv`), the cross-validation
pool (`dicts/cwik_trainval.dict`) and the WFST predictions this notebook compares against. Unlike the WFST notebooks, this one doesn't need
Phonetisaurus, but it does need `torch` and `transformers` (`pip install torch transformers`),
which the Docker image doesn't include. The model (1.2 GB) is downloaded once into the Hugging Face
cache. A GPU is used if there is one, but a CPU works too.

**The process at a glance**

1. **Load** the test split written by `Wik_eval.ipynb`.
2. **Load** the model from Hugging Face.
3. **Choose** the input format by testing a few options on part of the test set.
4. **Predict** every test word and save the predictions to `output/byt5_output.txt`.
5. **Score** the predictions with the same PER and WER code as `Wik_eval.ipynb`.
6. **Cross-validate** on the same 10 folds as `Wik_eval.ipynb` to check how stable the PER is.
7. **Compare** the result with the WFST model.
8. **Analyze** the errors, with the same breakdowns as `Wik_eval.ipynb`.

In [1]:
import os
import random
import sys
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, T5ForConditionalGeneration
from transformers.utils import logging as hf_logging

sys.dont_write_bytecode = True  # no __pycache__ in ../utils
sys.path.append(os.path.abspath("../utils"))
from tsv_metrics import tokenize_ipa, compute_per_wer, collect_errors, vowel_confusion
from alignment import align_trxn

hf_logging.set_verbosity_error()  # hide checkpoint warnings and progress bars
hf_logging.disable_progress_bar()

current_dir = Path().resolve()
dicts_dir   = current_dir / "dicts"
output_dir  = current_dir / "output"

/workspaces/Taglog-G2P/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load the test split

`Wik_eval.ipynb` saves its test split to `dicts/cwik_test.tsv`, one word per line with its phones
separated by spaces. The reference (`ref`) is built exactly as that notebook builds it, so both
models are scored against the same words and transcriptions.

The split has **2,172** entries and **2,171** unique words: one word is listed twice and keeps its
last transcription, as in `Wik_eval.ipynb`.

In [2]:
test_tsv = dicts_dir / "cwik_test.tsv"
if not test_tsv.is_file():
    raise FileNotFoundError(f"{test_tsv} not found. Run Wik_eval.ipynb first: it writes the test split.")

# Same reference Wik_eval scores against: the phones are already space-separated, and a word
# listed twice keeps its last transcription.
ref = {}
n_entries = 0
with open(test_tsv, encoding="utf-8") as f:
    for line in f:
        word, phones = line.rstrip("\n").split("\t")
        ref[word] = phones.split()
        n_entries += 1

print(f"Test entries     : {n_entries}")
print(f"Unique test words: {len(ref)}")

Test entries     : 2172
Unique test words: 2171


## 2. Load the model

The model and its tokenizer are downloaded from Hugging Face. `predict()` sends the words through
in batches of 64 and decodes greedily (one beam), the CharsiuG2P default.

The model's output needs two fixes before it can be compared with the references:

- It marks stress with an apostrophe (`ka'misa`), which `tokenize_ipa()` doesn't strip, so
  `to_phones()` removes it.
- It sometimes splits its output into several words (`naŋ ma`), so spaces are removed too.

`tokenize_ipa()` then splits the string into phones, the same way the references were made.

In [3]:
MODEL_ID = "lowestofthelow/filipino-byt5-g2p"
PREFIX   = "<fil>: "   # CharsiuG2P language tag (section 3)

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = T5ForConditionalGeneration.from_pretrained(MODEL_ID).to(device).eval()

print("Device:", device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")


def predict(words, prefix=PREFIX, add_special_tokens=True, batch_size=64):
    """Return {word: raw model output}, decoded greedily."""
    raw = {}
    for i in range(0, len(words), batch_size):
        batch = words[i:i + batch_size]
        enc = tokenizer([prefix + w for w in batch], padding=True,
                        add_special_tokens=add_special_tokens, return_tensors="pt").to(device)
        with torch.no_grad():
            out = model.generate(**enc, num_beams=1, max_new_tokens=128)
        raw.update(zip(batch, tokenizer.batch_decode(out, skip_special_tokens=True)))
    return raw


def to_phones(ipa):
    """Tokenize model output like the Wik_eval references: no spaces, no ' stress marks."""
    return tokenize_ipa("".join(ipa.split()).replace("'", ""))


def read_output(path):
    """Read a word TAB phones file into {word: [phones]}."""
    preds = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) >= 2:
                preds[parts[0]] = parts[1].split()
    return preds

Device: cpu
Parameters: 301M


## 3. Choose the input format

CharsiuG2P expects each word to start with a language tag, like `<eng-us>: hello`, but the model
card doesn't say which tag this fine-tune was trained with. So a few options are scored on the
first 300 test words:

| Option | What it tests |
| --- | --- |
| `<fil>: ` | Filipino tag |
| `<fil>: ` without the end-of-sequence token | the CharsiuG2P usage example, which leaves it out |
| `<tgl>: ` | Tagalog tag |
| no prefix | the bare word |

`<fil>: ` with the end-of-sequence token works best, with a PER of **1.70%** and a WER of
**9.00%** on the probe words, so it is the format used from here on. Leaving out the token more
than doubles the PER, and the other two options fail outright: `<tgl>: ` makes the model add a
stray word (`ʔet ka'misa`), and the bare word gives a PER of 44.70%.

In [4]:
probe_words = list(ref)[:300]
probe_ref = {w: ref[w] for w in probe_words}

rows = []
for prefix, special in [("<fil>: ", True), ("<fil>: ", False), ("<tgl>: ", True), ("", True)]:
    raw = predict(probe_words, prefix=prefix, add_special_tokens=special)
    m = compute_per_wer(probe_ref, {w: to_phones(s) for w, s in raw.items()})
    rows.append({
        "Prefix": repr(prefix),
        "End-of-sequence token": "yes" if special else "no",
        "PER (%)": round(m["per"] * 100, 2),
        "WER (%)": round(m["wer"] * 100, 2),
        "Example (kamisa)": predict(["kamisa"], prefix=prefix, add_special_tokens=special)["kamisa"],
    })

pd.DataFrame(rows)

,Prefix,End-of-sequence token,PER (%),WER (%),Example (kamisa)
0,'<fil>: ',yes,1.70,9.00,ka'misa
1,'<fil>: ',no,4.09,15.67,ka'misa
2,'<tgl>: ',yes,31.76,71.67,ʔet ka'misa
3,'',yes,44.70,43.33,ka'misa


## 4. Predict the test words

Every test word goes through the model with the chosen format. The predictions are saved to
`output/byt5_output.txt` in the same `word TAB phones` format as the WFST predictions.

All 2,171 words are predicted in a few minutes on a CPU. The examples show the fixes from
section 2 at work: the stress mark in `ka'misa` is dropped. They also show two kinds of error:
`kato` loses its word-final `ʔ`, the same ambiguity the WFST model faces, and `kutsada` gets the
affricate `t͡ʃ` where the reference has `t s`.

In [5]:
start = time.time()
raw = predict(list(ref))
hyp = {w: to_phones(s) for w, s in raw.items()}
print(f"Predicted {len(hyp)} words in {time.time() - start:.0f} s")

output_dir.mkdir(exist_ok=True)
byt5_output = output_dir / "byt5_output.txt"
with open(byt5_output, "w", encoding="utf-8") as f:
    for word, phones in hyp.items():
        f.write(f"{word}\t{' '.join(phones)}\n")

for word in ["kato", "kamisa", "alkaide", "kutsada"]:
    print(f"  {word:<10} raw:[{raw[word]}]  phones:[{' '.join(hyp[word])}]  ref:[{' '.join(ref[word])}]")

Predicted 2171 words in 198 s
  kato       raw:[ka'to]  phones:[k a t o]  ref:[k a t o ʔ]
  kamisa     raw:[ka'misa]  phones:[k a m i s a]  ref:[k a m i s a]
  alkaide    raw:[ʔalka'ʔide]  phones:[ʔ a l k a ʔ i d e]  ref:[ʔ a l k a j d e]
  kutsada    raw:[ku't͡ʃada]  phones:[k u t͡ʃ a d a]  ref:[k u t s a d a]


## 5. Score the predictions

PER and WER are computed with `compute_per_wer()`, the function `Wik_eval.ipynb` uses. An `assert`
checks that every test word got a prediction.

ByT5 reaches a PER of **1.64%** and a WER of **9.07%**: 1,974 of the 2,171 test words are fully
correct. Every test word got a prediction.

In [6]:
metrics = compute_per_wer(ref, hyp)

# Every test word should get a prediction.
assert metrics["n_words"] == len(ref), f"scored {metrics['n_words']} of {len(ref)} test words"

print("=== ByT5 (filipino-byt5-g2p) on the Wik_eval held-out test split ===")
print(f"  PER : {metrics['per']:.4f}  ({metrics['per'] * 100:.2f}%)   <- primary metric")
print(f"  WER : {metrics['wer']:.4f}  ({metrics['wer'] * 100:.2f}%)")
print(f"  Words evaluated  : {metrics['n_words']}")
print(f"  Correct words    : {metrics['n_correct']}")
print(f"  Total ref phones : {metrics['n_ref_phones']}")
print(f"  Total edit dist  : {metrics['n_edit']}")

=== ByT5 (filipino-byt5-g2p) on the Wik_eval held-out test split ===
  PER : 0.0164  (1.64%)   <- primary metric
  WER : 0.0907  (9.07%)
  Words evaluated  : 2171
  Correct words    : 1974
  Total ref phones : 17004
  Total edit dist  : 279


## 6. Cross-validation

The same 10 folds as section 5 of `Wik_eval.ipynb`: the 90% training and validation pool from
`dicts/cwik_trainval.dict`, shuffled with the same seed and cut into the same folds. There, the
WFST model is retrained on nine folds and tested on the tenth. ByT5 is already trained, so it
isn't retrained here: each round scores it on the held-out fold. The spread across the folds shows
how stable its PER is on different sets of words, next to the WFST's **1.03%** (± 0.12%).

This predicts all 19,549 words in the pool, about nine times the test split, so it takes a while
on a CPU. The predictions are saved to `output/byt5_cv_output.txt` and reused on the next run
(delete the file to predict again).

The mean PER across the folds is **1.54%** (± 0.11%). ByT5's PER is higher than the WFST's on
every fold, and it varies about as much from fold to fold.

In [7]:
trainval_dict = dicts_dir / "cwik_trainval.dict"
if not trainval_dict.is_file():
    raise FileNotFoundError(f"{trainval_dict} not found. Run Wik_eval.ipynb first: it writes the pool.")

# Rebuild Wik_eval's folds: the pool in file order, shuffled with the same seed.
trainval = []
with open(trainval_dict, encoding="utf-8") as f:
    for line in f:
        word, phones = line.rstrip("\n").split("\t")
        trainval.append((word, phones.split()))

random.seed(42)
cv_entries = trainval[:]
random.shuffle(cv_entries)

N_FOLDS = 10
fold_size = len(cv_entries) // N_FOLDS
folds = [cv_entries[i * fold_size:(i + 1) * fold_size] for i in range(N_FOLDS)]
folds[-1] += cv_entries[N_FOLDS * fold_size:]   # remainder into last fold
print("Fold sizes:", [len(f) for f in folds])

# ByT5 isn't retrained per fold, so each pool word is predicted once and shared by the folds.
cv_output = output_dir / "byt5_cv_output.txt"
cv_hyp = read_output(cv_output) if cv_output.is_file() else {}
todo = list(dict.fromkeys(w for w, _ in cv_entries if w not in cv_hyp))
if todo:
    start = time.time()
    cv_hyp.update({w: to_phones(s) for w, s in predict(todo).items()})
    print(f"Predicted {len(todo)} words in {time.time() - start:.0f} s")
    with open(cv_output, "w", encoding="utf-8") as f:
        for word, phones in cv_hyp.items():
            f.write(f"{word}\t{' '.join(phones)}\n")
else:
    print(f"Reused the predictions in {cv_output.name}")

fold_pers = []
for k, held_out in enumerate(folds):
    fold_ref = {w: p for w, p in held_out}
    m = compute_per_wer(fold_ref, cv_hyp)
    assert m["n_words"] == len(fold_ref), f"fold {k + 1}: scored {m['n_words']} of {len(fold_ref)} words"
    fold_pers.append(m["per"])
    print(f"  Fold {k + 1:2d}: PER {m['per'] * 100:5.2f}%  ({m['n_words']} words)")

fold_pers = np.array(fold_pers)
print()
print(f"10-fold CV PER: mean {fold_pers.mean() * 100:.2f}%  +/-  {fold_pers.std() * 100:.2f}%")

Fold sizes: [1954, 1954, 1954, 1954, 1954, 1954, 1954, 1954, 1954, 1963]
Reused the predictions in byt5_cv_output.txt
  Fold  1: PER  1.43%  (1954 words)
  Fold  2: PER  1.66%  (1954 words)
  Fold  3: PER  1.54%  (1954 words)
  Fold  4: PER  1.59%  (1954 words)
  Fold  5: PER  1.60%  (1954 words)
  Fold  6: PER  1.56%  (1954 words)
  Fold  7: PER  1.42%  (1954 words)
  Fold  8: PER  1.52%  (1954 words)
  Fold  9: PER  1.34%  (1954 words)
  Fold 10: PER  1.71%  (1963 words)

10-fold CV PER: mean 1.54%  +/-  0.11%


## 7. Compare with the WFST model

The WFST predictions in `output/cwik_output.txt` are scored against the same reference. That file
comes from `Wik_eval.ipynb`, so the table only has the WFST row after that notebook has run.

The WFST model is better on both metrics: a PER of **0.88%** against **1.64%**, and a WER of
**6.08%** against **9.07%**. ByT5 makes **279** phone edits to the WFST's 150, nearly twice as
many, and gets 65 fewer words fully correct.

In [8]:
rows = [{"Model": "ByT5 (filipino-byt5-g2p)", **metrics}]

cwik_output = output_dir / "cwik_output.txt"
if cwik_output.is_file():
    wfst = read_output(cwik_output)
    rows.insert(0, {"Model": "WFST (Wik_eval)", **compute_per_wer(ref, wfst)})
else:
    print(f"{cwik_output.name} not found: run Wik_eval.ipynb to add the WFST model to this table.")

summary = pd.DataFrame(rows)
summary["PER (%)"] = (summary["per"] * 100).round(2)
summary["WER (%)"] = (summary["wer"] * 100).round(2)
summary[["Model", "PER (%)", "WER (%)", "n_words", "n_correct", "n_ref_phones", "n_edit"]]

,Model,PER (%),WER (%),n_words,n_correct,n_ref_phones,n_edit
0,WFST (Wik_eval),0.88,6.08,2171,2039,17004,150
1,ByT5 (filipino-byt5-g2p),1.64,9.07,2171,1974,17004,279


## 8. Error analysis

The same breakdowns as in `Wik_eval.ipynb`: by error type, by phone and by substitution pair,
then the glottal stop and vowel-height checks.

### 8.1 Error types

There are **279** edits in total: 55 substitutions, 107 deletions and 117 insertions. Compared
with the WFST model (11, 64 and 75), ByT5 makes five times as many substitutions.

In [9]:
err = collect_errors(ref, hyp)
total_errors = err["n_sub"] + err["n_del"] + err["n_ins"]

pd.DataFrame([
    {"Type": "Substitution", "Count": err["n_sub"], "% of Errors": f"{err['n_sub'] / total_errors * 100:.1f}%"},
    {"Type": "Deletion",     "Count": err["n_del"], "% of Errors": f"{err['n_del'] / total_errors * 100:.1f}%"},
    {"Type": "Insertion",    "Count": err["n_ins"], "% of Errors": f"{err['n_ins'] / total_errors * 100:.1f}%"},
    {"Type": "TOTAL",        "Count": total_errors, "% of Errors": "100.0%"},
])

,Type,Count,% of Errors
0,Substitution,55,19.7%
1,Deletion,107,38.4%
2,Insertion,117,41.9%
3,TOTAL,279,100.0%


### 8.2 Phones behind each error type

The glottal stop `ʔ` is again the phone behind most deletions and insertions: it is dropped
**90** times and added **47** times. Substitutions are spread over more phones, led by `j` (16),
`t͡ʃ` (10) and `s` (9).

In [10]:
def top10(counter):
    return pd.DataFrame([{"Phone": p, "Count": c} for p, c in counter.most_common(10)])

print("Ref phones most often substituted:")
display(top10(err["sub_ref_counter"]))
print("Ref phones most often deleted:")
display(top10(err["del_counter"]))
print("Phones most often inserted:")
display(top10(err["ins_counter"]))

Ref phones most often substituted:


,Phone,Count
0,j,16
1,t͡ʃ,10
2,s,9
3,u,4
4,w,3
5,a,3
6,k,3
7,ɡ,2
8,ʔ,2
9,e,1


Ref phones most often deleted:


,Phone,Count
0,ʔ,90
1,s,7
2,t,3
3,e,2
4,u,2
5,a,1
6,w,1
7,k,1


Phones most often inserted:


,Phone,Count
0,ʔ,47
1,t,17
2,j,12
3,e,9
4,i,8
5,n,5
6,ɾ,5
7,w,5
8,a,3
9,ɡ,2


### 8.3 Substitution pairs

The most common pair is `t͡ʃ → s` (10), where the model reads the spelling *ts* as /ts/ and the
reference has the affricate /t͡ʃ/. Some pairs come from a different transcription convention
rather than a wrong sound: `j → ʃ` turns the *sy* in `seleksyon` into /ʃ/, and `j → i` writes a
glide as a full vowel (`keik` as /keʔik/). Others are real errors, such as `s → ʔ` at the start of
a word (`senyorita` as /ʔenjoɾita/).

In [11]:
display(pd.DataFrame([
    {"Pair": f"{r} -> {h}", "Count": c}
    for (r, h), c in err["sub_pair_counter"].most_common(10)
]))

print("Example words for the top substitution pairs:")
for (r, h), c in err["sub_pair_counter"].most_common(5):
    print(f"\n  {r} -> {h}  (count={c})")
    for ex in err["sub_pair_examples"].get((r, h), [])[:3]:
        print(f"    {ex['word']:<20}  ref:[{ex['ref']}]  hyp:[{ex['hyp']}]")

,Pair,Count
0,t͡ʃ -> s,10
1,j -> i,7
2,s -> ʔ,6
3,j -> ʃ,6
4,u -> w,3
5,s -> t͡ʃ,3
6,k -> ʔ,3
7,a -> o,2
8,w -> u,2
9,j -> e,1


Example words for the top substitution pairs:

  t͡ʃ -> s  (count=10)
    partse                ref:[p a ɾ t͡ʃ e]  hyp:[p a ɾ t s e]
    lakwatsero            ref:[l a k w a t͡ʃ e ɾ o]  hyp:[l a k w a t s e ɾ o]
    impatso               ref:[ʔ i m p a t͡ʃ o]  hyp:[ʔ i m p a t s o]

  j -> i  (count=7)
    keik                  ref:[k e j k]  hyp:[k e ʔ i k]
    taingang-baboy        ref:[t a j ŋ a ŋ b a b o j]  hyp:[t a ʔ i ŋ a ŋ b a b o j]
    alkaide               ref:[ʔ a l k a j d e]  hyp:[ʔ a l k a ʔ i d e]

  s -> ʔ  (count=6)
    senyorita             ref:[s e n j o ɾ i t a]  hyp:[ʔ e n j o ɾ i t a]
    sumugal               ref:[s u m u ɡ a l]  hyp:[ʔ i m u ɡ a l]
    sinasabunutan         ref:[s i n a s a b u n u t a n]  hyp:[ʔ i n a s a b u n u t a n]

  j -> ʃ  (count=6)
    ishinare              ref:[ʔ i s j i n e j ɾ]  hyp:[ʔ i ʃ i n a ɾ e]
    seleksyon             ref:[s e l e k s j o n]  hyp:[s e l e k ʃ o n]
    iniksyon              ref:[ʔ i n i k s j o n]  hyp:[ʔ i 

### 8.4 Phones the references never use

The model was trained on a different transcription convention, so it can produce phones that the
Wiktionary references never use. Each one counts as an error even if it is a reasonable sound.

Only **one** predicted phone never appears in the references: `ʒ`, once, in `bargen`
(/baɾdʒen/). So the convention differences in section 8.3 mostly swap one reference phone for
another, rather than bring in new symbols.

In [12]:
ref_inventory = {p for phones in ref.values() for p in phones}
hyp_counts = Counter(p for phones in hyp.values() for p in phones)
outside = {p: c for p, c in hyp_counts.most_common() if p not in ref_inventory}

print(f"Phones in the references : {len(ref_inventory)}")
print(f"Phones in the predictions: {len(hyp_counts)}")
print(f"Predicted phones never used in the references: {outside if outside else 'none'}")

for p in list(outside)[:5]:
    examples = [w for w, phones in hyp.items() if p in phones][:3]
    print(f"\n  {p}:")
    for w in examples:
        print(f"    {w:<20}  ref:[{' '.join(ref[w])}]  hyp:[{' '.join(hyp[w])}]")

Phones in the references : 26
Phones in the predictions: 27
Predicted phones never used in the references: {'ʒ': 1}

  ʒ:
    bargen                ref:[b a ɾ ɡ e n]  hyp:[b a ɾ d ʒ e n]


### 8.5 Where the glottal stop goes wrong

The glottal stop `ʔ` is the main weakness of the WFST model, because Tagalog spelling doesn't mark
it at the end of a word. This counts each `ʔ` deletion and insertion by its position in the word,
for both models.

ByT5 drops `ʔ` more often than the WFST model (**90** against 55) but adds it less often (**47**
against 69). Its errors are also less tied to the end of the word: **81.1%** of its deletions and
only **48.9%** of its insertions are word-final, against 90.9% and 88.4% for the WFST model. The
rest are medial or word-initial `ʔ` errors, which the WFST model rarely makes.

In [13]:
GLOTTAL = "ʔ"


def glottal_positions(ref, hyp):
    """Count ʔ deletions (by ref position) and insertions (by hyp position): initial/medial/final."""
    def bucket(idx, n):
        return "initial" if idx == 0 else "final" if idx == n - 1 else "medial"

    del_pos, ins_pos = Counter(), Counter()
    for word, ref_phones in ref.items():
        hyp_phones = hyp.get(word, [])
        aln_ref, aln_hyp, _, _ = align_trxn(ref_phones, hyp_phones)
        ri = hi = 0
        for a, b in zip(aln_ref, aln_hyp):
            if a == GLOTTAL and b == "*":
                del_pos[bucket(ri, len(ref_phones))] += 1
            if b == GLOTTAL and a == "*":
                ins_pos[bucket(hi, len(hyp_phones))] += 1
            ri += a != "*"
            hi += b != "*"
    return del_pos, ins_pos


models = {"ByT5": hyp}
if cwik_output.is_file():
    models = {"WFST": wfst, **models}

rows = []
for name, preds in models.items():
    del_pos, ins_pos = glottal_positions(ref, preds)
    for kind, counter in [("deleted", del_pos), ("inserted", ins_pos)]:
        rows.append({"Model": name, "ʔ": kind, **{p: counter[p] for p in ("initial", "medial", "final")},
                     "Total": sum(counter.values())})
pd.DataFrame(rows)

,Model,ʔ,initial,medial,final,Total
0,WFST,deleted,1,4,50,55
1,WFST,inserted,1,7,61,69
2,ByT5,deleted,4,13,73,90
3,ByT5,inserted,9,15,23,47


### 8.6 Vowel-height confusion: o↔u and e↔i

Tagalog *o*/*u* and *e*/*i* were historically allophones, so a model can swap them with no cue
from the spelling. `vowel_confusion()` counts every o↔u and e↔i substitution, in both directions.

There is just **one**: `mee` (/meʔeʔ/) becomes /miʔeʔ/, an e→i swap. There are no o↔u swaps,
so like the WFST model, ByT5 keeps these vowels apart.

In [14]:
vc = vowel_confusion(ref, hyp)   # default groups: o/u and e/i

display(pd.DataFrame([
    {"Confusion": k, "Count": v["count"], "% of subs": f"{v['rate'] * 100:.1f}%"}
    for k, v in vc.items()
]))

for label, v in vc.items():
    print(f"\n{label}: {v['count']} confusions")
    for (r, h), c in v["by_pair"].most_common():
        print(f"    {r} -> {h}: {c}")
    for ex in v["examples"][:5]:
        print(f"      {ex['word']:<18} ref:[{ex['ref']}]  hyp:[{ex['hyp']}]")

,Confusion,Count,% of subs
0,o/u,0,0.0%
1,e/i,1,1.8%



o/u: 0 confusions

e/i: 1 confusions
    e -> i: 1
      mee                ref:[m e ʔ e ʔ]  hyp:[m i ʔ e ʔ]
